In [3]:
print("OK")

OK


In [4]:
%pwd

'd:\\AI-Projects\\TciExpressChatBot\\research'

In [5]:
import os 
os.chdir(r"D:\AI-Projects\TciExpressChatBot")
%pwd

'D:\\AI-Projects\\TciExpressChatBot'

In [ ]:
%pwd

In [6]:
# from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyMuPDFLoader
    )
    return loader.load()

In [ ]:
extracted_data = load_pdf_files("data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

In [ ]:
texts_chunk

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
vector = embedding.embed_query("Hello world")
vector

In [ ]:
print( "Vector length:", len(vector))

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()


In [9]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GOOGLE_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [10]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec 

index_name = "chatbot-1"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# Add more data to the existing Pinecone index

In [ ]:
import os
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# -------------------------------------------------------------
# 1. TCI Express Comprehensive Knowledge Base Documents
# -------------------------------------------------------------
tci_documents = [
    # 1. Company Overview, History & Identity
    Document(
        page_content=(
            "Company Profile & Background: TCI Express Limited (tciexpress.in) is one of India's "
            "premier time-definite, multimodal B2B express logistics and delivery service providers. "
            "Originally founded in 1996 as 'TCI XPS' (a division of Transport Corporation of India), "
            "it was demerged in 2016 to form an independent, publicly listed company on the NSE and BSE "
            "(Ticker: TCIEXP). The company operates on an asset-light business model with over 3,500+ "
            "employees, an extensive hub-and-spoke distribution network, and specialized services in "
            "express cargo delivery and e-commerce distribution across India."
        ),
        metadata={
            "company": "TCI Express",
            "category": "overview",
            "section": "Company Profile and History",
            "source": "https://www.tciexpress.in/about-us",
        },
    ),
    # 2. Surface Express Service
    Document(
        page_content=(
            "Surface Express Service: TCI Express provides comprehensive time-bound surface express "
            "distribution across India. Features include door-to-door pickup and delivery, fully containerized "
            "all-weather GPS-enabled vehicles, and coverage across 40,000 to 60,000+ delivery locations. "
            "It is tailored for B2B Less-Than-Truckload (LTL) cargo and offers value-added options including "
            "Cash on Delivery (COD), Proof of Delivery (hard copy and digital E-POD), Delivery Against Consignee "
            "Copy (DACC), Out of Delivery Area (ODA) service, and holiday/Sunday deliveries."
        ),
        metadata={
            "company": "TCI Express",
            "category": "services",
            "service_type": "Surface Express",
            "source": "https://www.tciexpress.in",
        },
    ),
    # 3. Domestic & International Air Express
    Document(
        page_content=(
            "Air Express Services (Domestic and International): "
            "1. Domestic Air Express: 24-hour delivery in major metro cities and 48-hour delivery in tier-2/tier-3 "
            "cities for critical, high-value, and urgent consignments. Features late cut-off pick-up times, "
            "first-flight-out dispatch, and 24/7 client operations. "
            "2. International Air Express: Commercial and non-commercial air cargo connectivity serving over 208 "
            "countries globally from 3,000+ pickup points in India, handling door-to-door pickup, customs "
            "clearance assistance, cargo insurance facilitation, and third-country international logistics."
        ),
        metadata={
            "company": "TCI Express",
            "category": "services",
            "service_type": "Air Express",
            "source": "https://www.tciexpress.in",
        },
    ),
    # 4. Rail, C2C & Specialized Verticals
    Document(
        page_content=(
            "Multimodal, Rail & Specialized Express Offerings: "
            "1. Rail Express: Eco-friendly, cost-effective multimodal transport utilizing Indian Railways network "
            "for heavy, high-volume, bulk intercity cargo movement with station-to-station and door-to-door options. "
            "2. C2C Express (Customer to Customer / Full Truckload): Direct hub-to-hub and plant-to-customer dedicated "
            "movement for bulk commercial consignments. "
            "3. Pharma Cold Chain Express: Temperature-controlled transport catering to pharmaceuticals, vaccines, and "
            "healthcare products. "
            "4. Reverse Express: Dedicated reverse logistics for defective, returns, rental, and recycling goods."
        ),
        metadata={
            "company": "TCI Express",
            "category": "services",
            "service_type": "Rail and Specialized Logistics",
            "source": "https://www.tciexpress.in",
        },
    ),
    # 5. E-Commerce & Value-Added Services
    Document(
        page_content=(
            "E-Commerce Solutions and Value-Added Services: TCI Express supports fast-growing B2B e-commerce "
            "merchants and brand fulfillment. Services include first-mile pickup, mid-mile line haul, inventory "
            "movement between fulfillment centers, Return-to-Origin (RTO) management, Cash on Delivery (COD) "
            "handling, Freight on Delivery (FOD), electronic proof of delivery (E-POD), and SMS alerts."
        ),
        metadata={
            "company": "TCI Express",
            "category": "services",
            "service_type": "E-Commerce",
            "source": "https://www.tciexpress.in",
        },
    ),
    # 6. Technology, Tracking & Infrastructure
    Document(
        page_content=(
            "Technology & Infrastructure: TCI Express features 28+ large, strategically located sorting centers, "
            "including state-of-the-art fully automated sorting hubs in Gurugram and Pune equipped with high-speed "
            "automated conveyor belts. Tracking capabilities include real-time consignment tracking via airway bill / "
            "docket number (LR number) on tciexpress.in, tracking mobile applications, and REST API integration for "
            "enterprise ERP systems to automate order booking, manifesting, and delivery status updates."
        ),
        metadata={
            "company": "TCI Express",
            "category": "technology_and_tracking",
            "section": "Infrastructure and IT Systems",
            "source": "https://www.tciexpress.in",
        },
    ),
    # 7. Contact Details & Corporate Office
    Document(
        page_content=(
            "Corporate Office Contact and Support Details: "
            "Headquarters: TCI House, 69 Institutional Area, Sector 32, Gurugram - 122001, Haryana, India. "
            "Official Website: https://www.tciexpress.in | Primary Email: info@tciexpress.in. "
            "Contact Numbers: +91-124-2384090 / +91-124-2384094. "
            "Customer Support: Dedicated customer care divisions across 10 regional offices with online tools "
            "for pickup requests, pin code serviceability search, rate estimation, and branch locator."
        ),
        metadata={
            "company": "TCI Express",
            "category": "contact_and_support",
            "section": "Contact Information",
            "source": "https://www.tciexpress.in",
        },
    ),
]

# -------------------------------------------------------------
# 2. Upload Documents to Pinecone Vector Database
# -------------------------------------------------------------
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# OpenAI standard embedding model (1536 dimensions)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

index_name = "logistics-kb"

# Pinecone me upsert
vector_store = PineconeVectorStore.from_documents(
    documents=tci_documents,
    embedding=embeddings,
    index_name=index_name,
    namespace="tciexpress",
)

print("TCI Express ka poora knowledge base Pinecone me successfully load ho gaya!")

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'content-type': 'application/json', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'a63d092c41a5cb946263bf90270c169c', 'date': 'Mon, 07 Sep 2026 11:39:02 GMT', 'server': 'Google Frontend', 'Content-Length': '77', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"UNAUTHENTICATED","message":"Invalid API key"},"status":401}


In [ ]:
docsearch.add_documents(documents=[dswith])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Constructors?")
retrieved_docs

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7
)

# Test run
response = chatModel.invoke("Hello, are you ready?")
print(response.content)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Constructors?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is Java Package?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is the Instance variables?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what Local variables?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "Hello java"})

# Pinecone se chunks aaye ya nahi check karein:
retrieved_docs = response.get("context", [])

print(f"Total chunks retrieved from Pinecone: {len(retrieved_docs)}")

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} ---")
    print("Metadata (File & Page):", doc.metadata)
    print("Exact Content:", doc.page_content[:200])